<a href="https://colab.research.google.com/github/25wh1a0516-prog/NASSCOM/blob/main/U8_Data_Cleaning_Part2_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)


Setup complete. pandas 2.2.3


In [2]:
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df


,city,size,age,income,bought
0,pune,small,25,38000,0
1,delhi,large,41,92000,1
2,mumbai,medium,33,55000,0
3,pune,medium,29,47000,0
4,delhi,small,52,120000,1
5,mumbai,large,38,76000,1
6,pune,large,46,88000,1
7,delhi,small,22,41000,0


In [3]:
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe


,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


In [4]:
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]


,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [5]:
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)
m = MinMaxScaler().fit_transform(num)

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))


Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))


train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [8]:
fe = df.copy()

fe['income_per_age'] = (fe['income'] / fe['age']).round(0)

fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]




,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


In [9]:
dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d



,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']


pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])


In [11]:
X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)
acc = pipe.score(X_test, y_test)
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')


Test accuracy: 1.0
(small toy dataset — the point is the leak-free workflow, not the score)


In [13]:
#Lab Exercise 1:
# 1. One-hot encode the 'city' column of the whole DataFrame
df = pd.get_dummies(df, columns=['city'])

# 2. Ordinal-encode 'size' using size_order
size_code = {size: i for i, size in enumerate(size_order)}
df['size'] = df['size'].map(size_code)

print(df)

# 3. Explanation:
# One-hot encoding is wrong for size because size has a natural order
# such as Small, Medium, and Large.
# Ordinal encoding is wrong for city because cities have no natural order.

   size  age  income  bought  size_code  city_delhi  city_mumbai  city_pune
0     0   25   38000       0          0       False        False       True
1     2   41   92000       1          2        True        False      False
2     1   33   55000       0          1       False         True      False
3     1   29   47000       0          1       False        False       True
4     0   52  120000       1          0        True        False      False
5     2   38   76000       1          2       False         True      False
6     2   46   88000       1          2       False        False       True
7     0   22   41000       0          0        True        False      False


In [14]:
#Lab Exercise 2:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

income = df[['income']]

# 1. Standardise the income column
scaler = StandardScaler()
standard_income = scaler.fit_transform(income)

print("Standardised Mean:", standard_income.mean())
print("Standardised Std:", standard_income.std())

# 2. Min-Max scale the income column
minmax = MinMaxScaler()
scaled_income = minmax.fit_transform(income)

print("Min-Max Minimum:", scaled_income.min())
print("Min-Max Maximum:", scaled_income.max())

# 3. Needs scaling: K-Nearest Neighbors (KNN)
# Doesn't need scaling: Decision Tree


Standardised Mean: -2.7755575615628914e-17
Standardised Std: 1.0
Min-Max Minimum: 0.0
Min-Max Maximum: 1.0000000000000002


In [16]:
#Lab Exercise 3:
from sklearn.preprocessing import StandardScaler

# 1a. WRONG: Fit on ALL data
wrong_mean = StandardScaler().fit(X[num_cols]).mean_

# 1b. RIGHT: Fit on TRAIN data only
right_mean = StandardScaler().fit(X_train[num_cols]).mean_

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem:
# It causes data leakage because information from the test data
# influences the scaling process, leading to an unreliable evaluation.

fit-on-all mean : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]


In [17]:
#Lab Exercise 4:
ex = df.copy()

# 1. Create high_earner flag
ex['high_earner'] = ex['income'] > ex['income'].median()

# 2. Bin income into 3 buckets
ex['income_group'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['low', 'mid', 'high']
)

# 3. Show income and the new columns
print(ex[['income', 'high_earner', 'income_group']])


   income  high_earner income_group
0   38000        False          low
1   92000         True          mid
2   55000        False          low
3   47000        False          low
4  120000         True         high
5   76000         True          mid
6   88000         True          mid
7   41000        False          low


In [18]:
#Lab Exercise 5:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

Xn = df[['age', 'income']]
yn = df['bought']

# 1. Pipeline: MinMaxScaler -> LogisticRegression
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression())
])

# 2. Split data and fit on training set
X_train, X_test, y_train, y_test = train_test_split(
    Xn, yn, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)

# 3. Print test accuracy
print("Test Accuracy:", pipe.score(X_test, y_test))


Test Accuracy: 0.0
